# 01 – Business Understanding
**Project 2 – Ensemble learning for disease prediction (classification)**
Methodology: **CRISP-DM** · Reference article: Mahajan et al., *A comparative evaluation of machine learning ensemble approaches for disease prediction using multiple datasets*, Health Technology 14, 597–613 (2024). https://doi.org/10.1007/s12553-024-00835-w

This notebook covers the first CRISP-DM phase. It states the problem, the objectives, the success criteria, the scope, the risks and the plan. The next phases are documented in `02_eda.ipynb` (Data understanding) and `03_models_protocol.ipynb` (Data preparation, Modeling, Evaluation).

## 1. Context and problem statement

Early and reliable detection of chronic diseases (heart disease, diabetes, liver and kidney disorders, skin cancer) reduces complications and treatment costs. Hospitals and screening programmes collect structured records (laboratory results, symptoms, demographic variables), and machine learning can turn them into a **decision-support score** that helps clinicians prioritise patients for further examination.

A single decision tree is easy to read but unstable: a small change in the training data can produce a very different tree. **Ensemble methods** combine many models to reduce variance (Bagging, Random Forest), bias (Boosting) or both (Stacking). The reference article compares these families on **16 public medical datasets**. The practical question is:

> *Which ensemble approach predicts disease most reliably, and does the answer depend on the dataset, or does one method win consistently?*

The project answers this question by reproducing the comparative study with a transparent and reproducible protocol.

## 2. Business objectives

| # | Objective | Why it matters |
|---|---|---|
| B1 | Identify the ensemble technique that classifies disease presence most reliably across heterogeneous medical data | Gives practitioners an evidence-based default choice |
| B2 | Quantify how the ranking of methods changes from one disease and one dataset to another | A method that wins on one dataset can lose on another; users need to know how stable a recommendation is |
| B3 | Minimise missed diagnoses (false negatives) | Missing a sick patient is clinically more serious than ordering an unnecessary follow-up test |
| B4 | Produce results that another team can reproduce | Scientific credibility; comparison with the reference article |
| B5 | Report the limits of the data honestly (duplicates, small samples, imbalance) | Prevents over-optimistic conclusions |

## 3. Data mining goals

The business objectives are translated into technical tasks.

| Business objective | Data mining goal |
|---|---|
| B1, B2 | Train and compare **Decision Tree (baseline), Bagging, Random Forest, Extra Trees, AdaBoost, Gradient Boosting, XGBoost and Stacking** on every dataset, using the same data splits for all models |
| B3 | Report **Recall** (sensitivity) and **F1-score** next to Accuracy, and analyse confusion matrices (false negatives) |
| B4 | Fix the random seed (42), keep all settings in `configs/config.yaml`, use relative paths, provide unit tests and a `requirements.txt` |
| B5 | Audit each dataset (class balance, missing values, duplicates, leakage between train and test) and run the evaluation under two protocols: **naive** (data as downloaded) and **clean** (exact duplicates removed) |

**Task type:** supervised classification. Fourteen datasets are binary (disease = 1, no disease = 0). HAM10000 (D15) has 7 diagnostic classes and Dermatology (D16) has 6 classes in the original data, so multiclass metrics (macro-averaged) are required for them.

## 4. Success criteria

**Technical criteria**

* The four metrics requested by the project are computed for every model and every dataset: **Accuracy, Precision, Recall, F1-score**. ROC-AUC and AUPRC are added as complementary, threshold-independent measures.
* Scores come from **repeated stratified 5-fold cross-validation (3 repeats)**, reported as mean and standard deviation, not from one lucky split.
* Models are ranked per dataset, then compared by **average rank** across datasets. A difference is called meaningful only when it exceeds the fold-to-fold variability.
* Every ensemble is compared with the single decision tree and with the **majority-class baseline**. A model that does not beat the majority rule is reported as having learned nothing useful.
* Results are compared with Table 2 of the reference article. Differences (dataset size, duplicates, protocol) are explained, not hidden.

**Business criteria**

* The recommended method has high Recall without a collapse in Precision, so that the number of missed patients stays low and the number of false alarms stays acceptable.
* Conclusions are stated per dataset and overall, with their uncertainty.

**Choice of metrics**

| Metric | Reading in a medical setting |
|---|---|
| Accuracy | Share of correct decisions. Misleading when classes are unbalanced (for example ILPD: 71 % patients) |
| Precision | Among patients flagged as ill, the share who really are. Low precision means unnecessary examinations |
| **Recall** | Among truly ill patients, the share detected. Low recall means missed diagnoses |
| F1-score | Harmonic mean of Precision and Recall, a single summary of both |
| ROC-AUC / AUPRC | Quality of the probability ranking, independent of the decision threshold |

## 5. Scope: the 16 datasets of the study

The inventory below follows Table 2 of the reference article. *Rows* and *features* are the values reported by the article; the actual counts found in the files are verified in `02_eda.ipynb`.

In [1]:
import pandas as pd
from IPython.display import display
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)

inventory = pd.DataFrame([
    ("D1",  "Heart Disease (Comprehensive)",        "Heart",            "Kaggle", 1190, 11,  "binary"),
    ("D2",  "Heart Disease (Cleveland)",            "Heart",            "UCI",     303, 13,  "binary"),
    ("D3",  "Heart Disease (johnsmith88)",          "Heart",            "Kaggle", 1025, 13,  "binary"),
    ("D4",  "Heart Failure Clinical Records",       "Heart",            "Kaggle",  299, 12,  "binary"),
    ("D5",  "Liver Disorders (BUPA)",               "Liver",            "UCI",     345,  7,  "binary"),
    ("D6",  "Indian Liver Patient (ILPD)",          "Liver",            "UCI",     583, 10,  "binary"),
    ("D7",  "COVID-19 effect on liver cancer",      "Liver / cancer",   "Kaggle",  450, 25,  "binary"),
    ("D8",  "Early-stage diabetes risk",            "Diabetes",         "UCI",     520, 16,  "binary"),
    ("D9",  "Diabetes (Pima)",                      "Diabetes",         "Kaggle",  768,  7,  "binary"),
    ("D10", "Diabetes Dataset 2019",                "Diabetes",         "Kaggle",  952, 17,  "binary"),
    ("D11", "Diabetic Retinopathy Debrecen",        "Diabetes / eye",   "UCI",    1151, 18,  "binary"),
    ("D12", "Chronic Kidney Disease (UCI)",         "Kidney",           "UCI",     400, 25,  "binary"),
    ("D13", "Chronic Kidney Disease (Kaggle)",      "Kidney",           "Kaggle",  400, 13,  "binary"),
    ("D14", "Kidney Stone (urine analysis)",        "Kidney",           "Kaggle",   90,  7,  "binary"),
    ("D15", "Skin Cancer MNIST HAM10000 (metadata)","Skin cancer",      "Kaggle", 10015,  6, "multiclass (7)"),
    ("D16", "Dermatology",                          "Skin",             "UCI",     366, 35,  "multiclass (6)"),
], columns=["ID", "Dataset", "Disease area", "Source", "Rows (article)", "Features (article)", "Task"])

inventory

,ID,Dataset,Disease area,Source,Rows (article),Features (article),Task
0,D1,Heart Disease (Comprehensive),Heart,Kaggle,1190,11,binary
1,D2,Heart Disease (Cleveland),Heart,UCI,303,13,binary
2,D3,Heart Disease (johnsmith88),Heart,Kaggle,1025,13,binary
3,D4,Heart Failure Clinical Records,Heart,Kaggle,299,12,binary
4,D5,Liver Disorders (BUPA),Liver,UCI,345,7,binary
5,D6,Indian Liver Patient (ILPD),Liver,UCI,583,10,binary
6,D7,COVID-19 effect on liver cancer,Liver / cancer,Kaggle,450,25,binary
7,D8,Early-stage diabetes risk,Diabetes,UCI,520,16,binary
8,D9,Diabetes (Pima),Diabetes,Kaggle,768,7,binary
9,D10,Diabetes Dataset 2019,Diabetes,Kaggle,952,17,binary



**Interpretation of the inventory**

* **Coverage.** The study contains **16 datasets**: 9 from Kaggle (D1, D3, D4, D7, D9, D10, D13, D14, D15) and 7 from UCI (D2, D5, D6, D8, D11, D12, D16).
* **Task type.** **14 datasets are binary** (disease present or absent) and **2 are multiclass**: D15 (7 skin-lesion diagnoses) and D16 (6 dermatological diseases). Precision, Recall and F1 therefore need a macro-average for these two datasets, while the binary datasets use the disease class as the positive class.
* **Sample size varies by a factor of more than 100.** The smallest dataset is D14 (90 rows), the largest is D15 (10,015 rows). Half of the datasets have fewer than 500 rows (D14, D4, D2, D5, D16, D12, D13, D7), where cross-validation scores will fluctuate noticeably from fold to fold. This is the main reason why the evaluation uses 3 repetitions of 5-fold cross-validation rather than a single split.
* **Dimensionality varies from 6 to 35 features.** D16 (35 features) and D7 and D12 (25 features each) are rich, while D5, D9 and D14 have only 7 features. Methods that rely on random feature selection (Random Forest, Extra Trees) can behave differently in these two situations.
* **Repeated disease families.** Several datasets describe the same disease from different sources (four heart datasets, three kidney datasets, four diabetes-related datasets). This allows a within-disease comparison: if a method wins on all datasets of one disease, the result is less likely to be an accident of one file.


In [2]:
summary = (inventory.groupby("Disease area", sort=False)
           .agg(datasets=("ID", lambda s: ", ".join(s)), n=("ID", "count"), total_rows=("Rows (article)", "sum")))
summary

,datasets,n,total_rows
Disease area,,,
Heart,"D1, D2, D3, D4",4,2817
Liver,"D5, D6",2,928
Liver / cancer,D7,1,450
Diabetes,"D8, D9, D10",3,2240
Diabetes / eye,D11,1,1151
Kidney,"D12, D13, D14",3,890
Skin cancer,D15,1,10015
Skin,D16,1,366



**Interpretation of the summary by disease area**

* The 16 datasets add up to **18,857 rows**, but the volume is very unevenly distributed: **D15 alone represents 53 % of all rows** (10,015 of 18,857). Without D15, the remaining 15 datasets contain 8,842 rows, an average of about 590 rows per dataset.
* This imbalance is a reason to **average across datasets and not across rows**. If rows were pooled, D15 would dominate the conclusions and the 15 other datasets would barely count. Each dataset therefore receives the same weight in the ranking of methods.
* The area labels in the table are fine-grained (*Liver* and *Liver / cancer*, *Diabetes* and *Diabetes / eye*). The next cell groups them into five disease families to see the real structure of the study.


In [3]:

family = {"Heart": "Heart", "Liver": "Liver", "Liver / cancer": "Liver", "Diabetes": "Diabetes",
          "Diabetes / eye": "Diabetes", "Kidney": "Kidney", "Skin cancer": "Skin", "Skin": "Skin"}
inv = inventory.assign(Family=inventory["Disease area"].map(family))

by_family = (inv.groupby("Family", sort=False)
             .agg(datasets=("ID", lambda s: ", ".join(s)), n=("ID", "count"),
                  total_rows=("Rows (article)", "sum"),
                  smallest=("Rows (article)", "min"), largest=("Rows (article)", "max")))
by_family["share_of_rows_%"] = (100 * by_family["total_rows"] / by_family["total_rows"].sum()).round(1)
display(by_family)

rows = inventory["Rows (article)"]
print(f"datasets: {len(inventory)} | binary: {(inventory['Task'] == 'binary').sum()} | "
      f"multiclass: {(inventory['Task'] != 'binary').sum()}")
print(f"rows: min {rows.min()}, median {rows.median():.0f}, max {rows.max()}, total {rows.sum()}")
print(f"datasets with fewer than 500 rows: {(rows < 500).sum()} of {len(rows)}")
print(f"share of D15 in total rows: {100 * rows.max() / rows.sum():.1f} %")


,datasets,n,total_rows,smallest,largest,share_of_rows_%
Family,,,,,,
Heart,"D1, D2, D3, D4",4,2817,299,1190,14.9
Liver,"D5, D6, D7",3,1378,345,583,7.3
Diabetes,"D8, D9, D10, D11",4,3391,520,1151,18.0
Kidney,"D12, D13, D14",3,890,90,400,4.7
Skin,"D15, D16",2,10381,366,10015,55.1


datasets: 16 | binary: 14 | multiclass: 2
rows: min 90, median 485, max 10015, total 18857
datasets with fewer than 500 rows: 8 of 16
share of D15 in total rows: 53.1 %



**Interpretation by disease family**

* The study is organised around **five families**: heart (4 datasets), diabetes (4), liver (3), kidney (3) and skin (2). Heart and diabetes are the best covered, which makes conclusions about them more reliable than conclusions about skin diseases, where one of the two datasets is multiclass and the other is very large.
* In terms of volume the **skin family dominates** (more than half of all rows because of D15), followed by diabetes and heart. The kidney family is the smallest in rows (890) even though it has three datasets, because D14 has only 90 rows.
* **Dependence between datasets.** Datasets of the same family are not guaranteed to be independent. D12 and D13 describe the same chronic kidney disease cohort in two versions, and the heart datasets D1, D2 and D3 come from the same classic heart-disease collections, so some patients can appear in more than one of them. When results are averaged across datasets, this overlap makes the effective number of independent observations smaller than 16, and it will be mentioned when conclusions are drawn.
* **Consequence for the evaluation design.** The study mixes small and large samples, binary and multiclass tasks, and symptom questionnaires (D8), laboratory values (D6, D9, D12) and image-derived measurements (D11). A method that is best on one type of data may not be best on another, which is exactly why the comparison is run per dataset first and aggregated by mean rank afterwards.


**Heterogeneity is deliberate.** Sizes range from 90 to 10,015 rows, features from 6 to 35, and the data mix numeric laboratory values, binary symptoms and categorical variables. A method that is good on only one kind of data will show up in the cross-dataset ranking.

**In scope:** tabular data, classification, the ensemble families listed above, tuning under an equal budget for all models, statistical comparison across datasets.
**Out of scope:** medical images (only the HAM10000 metadata table is used), deep learning, clinical deployment, causal claims.

## 6. Methods under study

| Family | Principle | Models in the project | Main effect |
|---|---|---|---|
| Single model (baseline) | One tree grown on all data | Decision Tree (CART, also implemented from scratch in `notebooks/annex/` and verified against scikit-learn) | High variance, interpretable |
| Bagging | Trees trained on bootstrap samples, votes averaged | Bagging, Random Forest, Extra Trees | Reduces variance; Random Forest adds random feature subsets |
| Boosting | Models trained sequentially, each correcting the errors of the previous ones | AdaBoost, Gradient Boosting, XGBoost | Reduces bias |
| Stacking | A meta-model learns how to combine heterogeneous base models | Stacking (Random Forest, AdaBoost, Gradient Boosting, XGBoost → Logistic Regression) | Exploits complementary errors |

## 7. Evaluation design

The design is fixed before looking at any result, so that all models are treated identically.

| Element | Choice |
|---|---|
| Resampling | Repeated stratified K-fold, 5 folds × 3 repeats (`configs/config.yaml`) |
| Reproducibility | Global seed 42; identical folds for every model |
| Preprocessing | Inside each fold only (imputation, one-hot encoding), so that no information from the test fold leaks into training |
| Protocol A – *naive* | Data used exactly as downloaded, duplicates kept |
| Protocol B – *clean* | Exact duplicate rows removed before splitting |
| Reported metrics | Accuracy, Precision, Recall, F1 (required); ROC-AUC, AUPRC (complementary); confusion matrices |
| Multiclass datasets | Macro-averaged Precision, Recall and F1 |
| Comparison across datasets | Mean rank of each model; comparison with Table 2 of the article |
| Tuning | Hyperparameter search with the same budget for each model, performed inside the cross-validation (nested) to avoid optimistic bias |

Running both protocols measures how much a naive evaluation **overestimates** performance when the same patient record appears in both the training and the test set.

## 8. Assumptions, constraints and risks

| Risk or constraint | Consequence | Mitigation |
|---|---|---|
| **Duplicate records** (very frequent in some files, for example the early-stage diabetes survey and the johnsmith88 heart file) | Identical rows in train and test inflate scores | Duplicate audit in the EDA; naive versus clean protocol |
| **Missing values** (for example CKD, about 10 % of cells) | Bias if handled carelessly | Imputation fitted on the training fold only |
| **Zeros that encode missing values** (for example cholesterol or glucose equal to 0) | Distorted distributions | Detected in the EDA and treated as missing |
| **Very small datasets** (Kidney Stone: 90 rows) | High variance, unstable ranking | Repeated cross-validation, report of standard deviations |
| **Class imbalance** (for example ILPD: 71 % positive) | Accuracy hides poor recall | Report Recall, F1, AUPRC; majority-class baseline |
| **Near-perfect predictors or trivially separable problems** (for example Dermatology binarised) | Little room to separate methods | Single-feature and default-model audit in the EDA |
| **Ambiguous target definitions** (BUPA `selector` is a train/test flag in the UCI description, not a diagnosis) | Results may not reflect a clinical label | Documented in the target registry and in the report |
| **Overlapping sources** (D12 and D13 describe the same chronic kidney disease cohort in two versions) | The two datasets are not independent observations | Stated explicitly when averaging across datasets |
| **Multiclass tasks** (D15, D16) | Binary metrics undefined | Macro-averaged metrics |
| **Differences from the article** (versions, preprocessing, undisclosed tuning) | Exact figures cannot be expected to match | Comparison of trends and ranks rather than identical numbers |
| **Not a clinical tool** | Models are trained on small public datasets without external validation | Results are presented as a methodological comparison only |

## 9. Ethical and practical considerations

* All datasets are public and de-identified. No attempt is made to re-identify patients.
* Predictive scores are intended to **support** a clinician's decision, never to replace it.
* Performance obtained on a public dataset does not transfer automatically to another hospital or population; external validation would be required before any real use.
* Class imbalance and unequal sample sizes can hide poor performance on minority groups; per-class metrics are therefore reported.

## 10. Tools and resources

| Need | Tool |
|---|---|
| Language | Python ≥ 3.9 |
| Data handling | pandas, NumPy |
| Models | scikit-learn, XGBoost |
| Visualisation | matplotlib, seaborn |
| Data access | `ucimlrepo` (UCI), manual download (Kaggle) |
| Quality assurance | pytest (unit tests for the data loader, the protocol and the CART implementation) |
| Reproducibility | `configs/config.yaml`, `requirements.txt`, `pyproject.toml`, fixed seed |

## 11. Project plan (CRISP-DM)

| Phase | Content | Where | Deliverable |
|---|---|---|---|
| 1. Business understanding | Problem, objectives, success criteria, risks, plan | `01_business_understanding.ipynb` (this notebook) | This document |
| 2. Data understanding | Collection of the 16 datasets, target definition, class balance, missing values, duplicates, leakage, baselines | `02_eda.ipynb`, `src/ensemble_disease/eda.py` | Tables and figures in `results/eda_*` |
| 3. Data preparation | Cleaning of text cells, missing-value imputation, one-hot encoding, duplicate removal, binarisation of D16, multiclass handling of D15 | `data.py`, `eda.make_preprocessor`, `protocol.prepare` | Reproducible pipeline |
| 4. Modeling | Eight models, hyperparameter tuning with an equal budget | `models.py`, `03_models_protocol.ipynb` | Tuned models |
| 5. Evaluation | Repeated stratified cross-validation, four required metrics, ranks, confusion matrices, comparison with the article | `protocol.py`, `03_models_protocol.ipynb` | `results/cv_results_long.csv`, summary tables |
| 6. Deployment (adapted) | Replaced by the written report and the presentation | PDF report, slides | Final report and PowerPoint |

CRISP-DM is iterative. Findings from the data understanding phase (for example the share of duplicates) feed back into the evaluation design, which is why two protocols are defined.